# TenantEase Khata model walkthrough

This notebook clones the public repository and reproduces the synthetic data, model training, and evaluation. Every tenant and payment record is synthetic. Results are not production accuracy.

In [ ]:
!git clone https://github.com/agniv-dutta/TenantEase.git
%cd TenantEase
!pip install -e 'backend[dev]'

## Generate the seeded portfolio and train

The seed and demo clock are fixed for reproducibility. Training uses the feature pipeline in the repository, including its as-of cutoff.

In [ ]:
%cd backend
!python -m app.ml.synth --seed 42 --out data/khata.db
!python -m app.ml.train
!python -m app.ml.evaluate

## Read the evaluation

The generated metrics are written to `app/ml/artifacts/metrics.json`. Compare the shipped calibrated model with the rule baseline, then read `../../docs/model_card.md` and `../../docs/eval_report.md` for split details, calibration, slices, uncertainty, and limitations. Synthetic test metrics do not predict performance on real tenants.

In [ ]:
import json
from pathlib import Path
metrics = json.loads(Path('app/ml/artifacts/metrics.json').read_text())
print('Rows:', metrics['n_test_rows'])
for name in ('B0_rule_baseline', metrics['shipped_model_key']):
    score = metrics['models'][name]
    print(name, {key: score[key] for key in ('pr_auc', 'roc_auc', 'brier')})